## DEMO WORD SIMILARITY


In [ ]:
!pip install gensim

`gensim.downloader` cho phép tải các dataset mẫu cho NLP.

Ở đây ta dùng corpus **text8**:
- Được trích từ Wikipedia
- Chỉ gồm chữ thường
- Thường dùng để demo Word2Vec


In [ ]:
import gensim.downloader as api

print(api.info()["corpora"].keys())

In [ ]:
dataset = api.load("text8")

In [ ]:
for i, sentence in enumerate(dataset):
    print(sentence[:10])
    if i == 2:
        break

### Word2Vec Model
Word2Vec học **dense embeddings** cho mỗi từ.

Một embedding:
- là vector số thực
- thường 50–300 chiều
- biểu diễn ý nghĩa của từ

Tham số quan trọng:
- `vector_size`: số chiều của embedding
- `window`: kích thước context window
- `min_count`: bỏ các từ quá hiếm


In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=dataset,
    vector_size=100,   # kích thước embedding
    window=5,          # context window
    min_count=5,       # bỏ các từ hiếm
    workers=4,
    sg=1,              # 1 = skip-gram
    epochs=5
)

print("Training complete")

### Most Similar Words
Hàm `most_similar()` tìm các từ gần nhất trong **embedding space**.

In [ ]:
model.wv.most_similar("king")

In [ ]:
words = ["king", "computer", "dog", "music"]

for w in words:
    print("\nWord:", w)
    print(model.wv.most_similar(w, topn=5))

### Word Similarity
`model.wv.similarity(a,b)` tính cosine similarity giữa hai từ.

Ví dụ:
- `king` và `queen` có similarity cao
- `king` và `banana` thì thấp


In [ ]:
print("Similarity king queen:",
      model.wv.similarity("king", "queen"))

print("Similarity king banana:",
      model.wv.similarity("king", "banana"))

In [ ]:
def show_similar_words(word, topn=10):

    if word not in model.wv:
        print("Word not in vocabulary")
        return

    similar = model.wv.most_similar(word, topn=topn)

    print(f"\nWords most similar to '{word}':")

    for w, score in similar:
        print(f"{w:15} similarity = {score:.4f}")

In [ ]:
show_similar_words("frog")
show_similar_words("king")
show_similar_words("computer")
show_similar_words("music")

### Word Analogy
Word2Vec có khả năng giải **analogy**:

king - man + woman ≈ queen

Ý tưởng:
vector(king) − vector(man) + vector(woman) ≈ vector(queen)


In [ ]:
model.wv.most_similar(
    positive=["king", "woman"],
    negative=["man"]
)

### Visualization
Embedding thường có **100–300 chiều**, nên khó trực quan hóa.

Ta dùng **t‑SNE** để:
- giảm chiều (dimensionality reduction)
- chiếu vector xuống 2D
- quan sát các cụm từ có nghĩa gần nhau


visualizing by t-SNE

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
def build_vector_dict(model, matrix=None):
    """
    Convert Word2Vec model thành dict: word -> vector
    """
    words = model.wv.index_to_key

    if matrix is None:
        matrix = model.wv.vectors

    return {w: matrix[i] for i, w in enumerate(words)}

In [ ]:
def plot_tsne_words(vec_dict, words, title="t-SNE Visualization"):
    valid_words = [w for w in words if w in vec_dict]

    if len(valid_words) < 2:
        print("Không đủ từ để vẽ.")
        return

    X = np.array([vec_dict[w] for w in valid_words])

    tsne = TSNE(
        n_components=2,
        perplexity=min(30, len(valid_words) - 1),
        random_state=42,
        init="pca",
        learning_rate="auto"
    )
    X_2d = tsne.fit_transform(X)

    plt.figure(figsize=(10, 8))
    plt.scatter(X_2d[:, 0], X_2d[:, 1])

    for i, word in enumerate(valid_words):
        plt.annotate(word, (X_2d[i, 0], X_2d[i, 1]), fontsize=10)

    plt.title(title)
    plt.xlabel("t-SNE dim 1")
    plt.ylabel("t-SNE dim 2")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
def plot_word_neighbors_tsne(vec_dict, target_word, topn=15):

    if target_word not in vec_dict:
        print(f"'{target_word}' không có trong vocabulary.")
        return

    target_vec = vec_dict[target_word]
    sims = []

    # tính cosine similarity
    for w, v in vec_dict.items():
        if w == target_word:
            continue
        sim = np.dot(target_vec, v) / (np.linalg.norm(target_vec) * np.linalg.norm(v))
        sims.append((w, sim))

    sims.sort(key=lambda x: x[1], reverse=True)
    neighbors = [w for w, _ in sims[:topn]]

    words = [target_word] + neighbors
    X = np.array([vec_dict[w] for w in words])

    tsne = TSNE(
        n_components=2,
        perplexity=min(30, len(words)-1),
        random_state=42,
        init="pca",
        learning_rate="auto"
    )

    X_2d = tsne.fit_transform(X)

    plt.figure(figsize=(10,8))

    # vẽ target
    plt.scatter(X_2d[0,0], X_2d[0,1], color="red", s=120)
    plt.annotate(target_word, (X_2d[0,0], X_2d[0,1]), fontsize=12, fontweight="bold")

    # vẽ neighbors
    for i, word in enumerate(neighbors, start=1):
        plt.scatter(X_2d[i,0], X_2d[i,1], color="blue")
        plt.annotate(word, (X_2d[i,0], X_2d[i,1]), fontsize=10)

    plt.title(f"t-SNE of '{target_word}' and neighbors")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
vecs = build_vector_dict(model)
plot_word_neighbors_tsne(vecs, "apple", topn=20)
plot_word_neighbors_tsne(vecs, "king", topn=20)

parallelogram model

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def parallelogram_analogy(model, a, b, c, topn=10):
    """
    Solve analogy: a : b :: c : ?
    Equivalent to: vec(b) - vec(a) + vec(c)

    Example:
        man : king :: woman : ?
        => a='man', b='king', c='woman'
    """

    # Nếu dùng gensim Word2Vec thì từ vựng nằm trong model.wv
    vectors = model.wv if hasattr(model, "wv") else model

    # Kiểm tra OOV
    for word in [a, b, c]:
        if word not in vectors:
            print(f"'{word}' not in vocabulary")
            return []

    # Vector theo parallelogram model
    target = vectors[b] - vectors[a] + vectors[c]

    results = []

    # Duyệt toàn bộ vocab để tìm từ gần nhất
    for word in vectors.key_to_index:
        if word in [a, b, c]:
            continue
        sim = cosine_similarity(target, vectors[word])
        results.append((word, sim))

    # Sắp xếp giảm dần theo độ giống
    results = sorted(results, key=lambda x: x[1], reverse=True)

    return results[:topn]

In [ ]:
def show_analogy(model, a, b, c, topn=5):
    results = parallelogram_analogy(model, a, b, c, topn=topn)
    if not results:
        return

    print(f"{a} : {b} :: {c} : ?")
    for word, score in results:
        print(f"{word:15s} {score:.4f}")

In [ ]:
show_analogy(model, "man", "king", "woman")
show_analogy(model, "france", "paris", "italy")

Evaluate

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("wordsim353crowd.csv")
print(df.head())

In [ ]:
from scipy.stats import spearmanr

human_scores = []
model_scores = []

for _, row in df.iterrows():

    w1 = row["Word 1"].lower()
    w2 = row["Word 2"].lower()
    human = float(row["Human (Mean)"])

    if w1 in model.wv and w2 in model.wv:
        sim = model.wv.similarity(w1, w2)

        human_scores.append(human)
        model_scores.append(sim)

In [ ]:
corr, p_value = spearmanr(human_scores, model_scores)

print("Number of valid pairs:", len(human_scores))
print("Number of OOV pairs:", len(oov_pairs))
print("Spearman correlation:", corr)
print("p-value:", p_value)